#### Etapa 2: Modelagem com Redes Neurais (PyTorch) e Ensembles

***Objetivos desta etapa:***
1. Preparar os dados em Tensores e DataLoaders (Batching).
2. Construir e treinar uma Rede Neural (MLP) usando PyTorch com Early Stopping e tratamento de desbalanceamento (`pos_weight`).
3. Treinar um modelo Ensemble de Árvore (Random Forest) como Baseline robusto.
4. Avaliar ambos com $\ge 4$ métricas e registrar no MLflow.
5. Analisar o Trade-off de Custo de Negócio (Falso Positivo vs Falso Negativo).

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sys.path.append(os.path.abspath(".."))
from src.utils.logger import get_logger
logger = get_logger("notebook_pytorch")

logger.info("1. Carregando e preparando dados para o PyTorch...")
df = pd.read_csv("../data/raw/telco_churn.csv")
df = df.drop(columns=['customerID'])
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(' ', ''), errors='coerce').fillna(0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df_ml = pd.get_dummies(df, drop_first=True)

X = df_ml.drop(columns=['Churn']).values.astype(np.float32)
y = df_ml['Churn'].values.astype(np.float32)

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

class TelcoDataset(Dataset):
    """Converte arrays NumPy para tensores do PyTorch."""
    def __init__(self, features, labels):
        self.X = torch.tensor(features)
        self.y = torch.tensor(labels).unsqueeze(1)

    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

BATCH_SIZE = 64
train_loader = DataLoader(TelcoDataset(X_train_scaled, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TelcoDataset(X_val_scaled, y_val), batch_size=BATCH_SIZE, shuffle=False)

logger.info(f"Setup concluído. Features de entrada: {X_train_scaled.shape[1]}")

In [ ]:
import torch.nn as nn
import torch.optim as optim

logger.info("2. Construindo a MLP e iniciando o treinamento...")

class ChurnMLP(nn.Module):
    """Arquitetura Multilayer Perceptron para classificação binária."""
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2), 
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1) 
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class EarlyStopping:
    """Interrompe o treino se a validação parar de melhorar."""
    def __init__(self, patience=7):
        self.patience = patience
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None or val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

# Instanciando o motor
model = ChurnMLP(input_dim=X_train_scaled.shape[1])
# pos_weight=2.8 ensina o modelo que errar o Churn é 2.8x "pior" que errar retenção
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([2.8]))
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Loop de Treinamento
early_stopping = EarlyStopping()
for epoch in range(100): # máximo de 100 épocas
    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
    
    # Validação (Para o Early Stopping)
    model.eval()
    val_loss = sum(criterion(model(X_b), y_b).item() for X_b, y_b in val_loader) / len(val_loader)
    
    early_stopping(val_loss)
    if early_stopping.early_stop:
        logger.info(f"Parada antecipada acionada na Época {epoch+1}!")
        break

logger.info("Treinamento da Rede Neural finalizado!")

In [ ]:
import mlflow
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score

logger.info("3. Avaliando modelos e salvando logs no MLflow...")

# 1. Previsões da Rede Neural (MLP)
model.eval()
with torch.no_grad():
    y_pred_logits = model(torch.tensor(X_test_scaled))
    y_pred_prob_mlp = torch.sigmoid(y_pred_logits).numpy().squeeze()
    y_pred_mlp = (y_pred_prob_mlp >= 0.5).astype(int)

mlp_metrics = {
    "accuracy": accuracy_score(y_test, y_pred_mlp),
    "f1_score": f1_score(y_test, y_pred_mlp),
    "roc_auc": roc_auc_score(y_test, y_pred_prob_mlp),
    "pr_auc": average_precision_score(y_test, y_pred_prob_mlp)
}

# 2. Treinamento e Previsões do Ensemble (Random Forest)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
rf_model.fit(X_train_scaled, y_train) # Random Forest também recebe os dados escalonados aqui para manter igualdade

y_pred_rf = rf_model.predict(X_test_scaled)
y_pred_prob_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

rf_metrics = {
    "accuracy": accuracy_score(y_test, y_pred_rf),
    "f1_score": f1_score(y_test, y_pred_rf),
    "roc_auc": roc_auc_score(y_test, y_pred_prob_rf),
    "pr_auc": average_precision_score(y_test, y_pred_prob_rf)
}

# 3. Registrando no MLflow
mlflow.set_experiment("Telco_Churn_Etapa2")

with mlflow.start_run(run_name="PyTorch_MLP_Final"):
    mlflow.log_params({"epochs_run": epoch+1, "batch_size": BATCH_SIZE, "pos_weight": 2.8})
    mlflow.log_metrics(mlp_metrics)
    
with mlflow.start_run(run_name="RandomForest_Ensemble"):
    mlflow.log_params({"n_estimators": 100, "class_weight": "balanced"})
    mlflow.log_metrics(rf_metrics)

logger.info("Métricas salvas no MLflow com sucesso!")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

logger.info("4. Gerando Matrizes de Confusão para análise de custo...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz MLP
sns.heatmap(confusion_matrix(y_test, y_pred_mlp), annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_title("MLP PyTorch (Rede Neural)")
axes[0].set_xlabel("Previsão (1=Churn, 0=Fica)")
axes[0].set_ylabel("Realidade")

# Matriz Random Forest
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt="d", cmap="Greens", ax=axes[1])
axes[1].set_title("Random Forest (Ensemble)")
axes[1].set_xlabel("Previsão (1=Churn, 0=Fica)")
axes[1].set_ylabel("Realidade")

plt.tight_layout()
plt.show()

logger.info("Etapa 2 finalizada, analise o quadrante inferior esquerdo (Falsos Negativos).")